# Tutorial 2 — JSON / document retrieval with a JSON-path index

Store semi-structured documents in a `Utf8` JSON column, add a **JSON-path
inverted index** over the paths you filter on, and query them with the
PostgreSQL-style `json_*` functions. The index is a rebuildable overlay — the
Parquet data is never rewritten.

See `docs/tutorials/02_json_document_retrieval.md` for the SQL/connector form.

In [ ]:
import json
import shutil
import pyarrow as pa
import benostreamdb as bsdb

shutil.rmtree("/tmp/bsdb_tutorial2", ignore_errors=True)

docs = [
    {"id": 1, "doc": json.dumps({"user": {"id": "u-42"}, "level": "gold"})},
    {"id": 2, "doc": json.dumps({"user": {"id": "u-7"}, "level": "silver"})},
    {"id": 3, "doc": json.dumps({"user": {"id": "u-42"}, "level": "silver"})},
]
table = bsdb.Table.from_arrow(
    "file:///tmp/bsdb_tutorial2/docs",
    pa.table({"id": [d["id"] for d in docs], "doc": [d["doc"] for d in docs]}),
)
table.to_pandas()

## Add the JSON-path index

Index the paths you filter on. Each configured path becomes a
`(path, value) -> row_ids` overlay packed into the segment's Puffin bundle.

In [ ]:
table.add_index("doc", {"type": "json_path", "paths": ["$.user.id", "$.level"]})
print("indexed columns:", table.index_columns)

## Query with the `json_*` functions

Equality on an indexed path (`json_extract_path_text(...) = 'value'`) and
containment (`json_contains`) are rewritten into index lookups; unindexed paths
fall back to a full scan.

In [ ]:
session = bsdb.Session()
session.register("docs", table)

# Equality on an indexed path. The path is *variadic* (one arg per level).
print(session.sql(
    "SELECT id FROM docs WHERE json_extract_path_text(doc, 'user', 'id') = 'u-42'"
))

# Containment (recursive @>).
print(session.sql(
    "SELECT id FROM docs WHERE json_contains(doc, '{\"level\": \"silver\"}')"
))

# Extract a value.
print(session.sql("SELECT id, json_extract_path_text(doc, 'level') AS level FROM docs"))

## Why this matters

The index is an **advisory overlay**: if it is dropped or lost, the same query
still returns the correct rows via a full scan, and the overlay rebuilds. That
is the story — *"add an index to an existing Iceberg table without rewriting
the table."*